## Setup del entorno para la API de Gemini

Para usar la API de Gemini, necesitarás una clave API. Si aún no tienes una, créala en [Google AI Studio](https://aistudio.google.com/app/apikey). Es importante mantener tu clave API segura.

En Colab, puedes añadir la clave al gestor de secretos (el icono de la "🔑" en el panel izquierdo). Dale el nombre `GOOGLE_API_KEY`. Luego pasaremos la clave al SDK.

**Nota sobre la API key:** este notebook corre fuera de Google Colab, así que en vez del gestor de secretos de Colab (`userdata.get`), la clave se lee desde un archivo `.env` en la **raíz del repo** (`agentic-evals/.env`, no en este módulo) usando `python-dotenv`. Copiá `.env.example` a `.env` en la raíz y completá `GEMINI_API_KEY` con tu clave de [Google AI Studio](https://aistudio.google.com/app/apikey) antes de ejecutar la siguiente celda.

In [ ]:
# Importa el SDK de Python
import os

import google.generativeai as genai
from dotenv import load_dotenv

# La clave se lee desde .env en la raíz del repo (python-dotenv la busca
# subiendo desde el cwd de este notebook), no desde el gestor de secretos de Colab.
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

Una vez configurada la clave API, necesitamos inicializar el modelo generativo. Usaremos `gemini-pro` para empezar, ya que es un modelo de texto generalista.

In [ ]:
# Inicializa el modelo Gemini
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')
print("Modelo Gemini inicializado con éxito!")

In [ ]:
from pathlib import Path

PROMPTS_DIR = Path.cwd() / "prompts"


def load_prompt(name: str) -> str:
    """Carga el texto de un prompt desde prompts/{name}.md (relativo al cwd del notebook)."""
    return (PROMPTS_DIR / f"{name}.md").read_text(encoding="utf-8").strip()

Ahora que el modelo está configurado, podemos empezar a interactuar con él para observar cómo se comporta y, específicamente, buscar ejemplos de alucinaciones.

## Observando Alucinaciones del Modelo Gemini

Las alucinaciones en los LLM ocurren cuando el modelo genera información que es falsa, inventada o no está respaldada por los datos de entrenamiento, pero la presenta como un hecho. Vamos a intentar provocar algunas de estas situaciones.

### Ejemplo 1: Pregunta sobre hechos oscuros o inexistentes

Un método común para observar alucinaciones es pedirle al modelo información sobre personas o eventos muy oscuros, o incluso inventados, para ver si el modelo inventa una respuesta convincente.

In [ ]:
prompt_1 = load_prompt("nobel_literatura_1890")
response_1 = gemini_model.generate_content(prompt_1)

print(f"Pregunta: {prompt_1}\n")
print(f"Respuesta del modelo:\n{response_1.text}")

# Nota: El Premio Nobel de Literatura se estableció en 1901. Cualquier mención a un ganador en 1890 sería una alucinación.

### Ejemplo 2: Creación de detalles inexistentes para un objeto conocido

Otro tipo de alucinación puede ocurrir cuando el modelo inventa características o detalles sobre un tema conocido, pero que no son reales.

In [ ]:
prompt_2 = load_prompt("uss_enterprise_d_propulsion")
response_2 = gemini_model.generate_content(prompt_2)

print(f"Pregunta: {prompt_2}\n")
print(f"Respuesta del modelo:\n{response_2.text}")

# Nota: Si bien la USS Enterprise D es real en Star Trek, el 'Volumen 5 de la Guía Técnica' puede ser un invento, o los 'tres principales sistemas de propulsión' con esa especificidad podrían ser detalles inventados por el LLM.

### Ejemplo 3: Preguntas con premisas falsas

Pedir al modelo que elabore sobre una premisa que es intrínsecamente incorrecta o falsa puede llevarlo a "aceptar" la falsedad y luego alucinar detalles para apoyarla.

In [ ]:
prompt_3 = load_prompt("julio_cesar_luna")
response_3 = gemini_model.generate_content(prompt_3)

print(f"Pregunta: {prompt_3}\n")
print(f"Respuesta del modelo:\n{response_3.text}")

# Nota: Julio César nunca fue a la Luna. El modelo podría alucinar completamente estrategias y detalles de una colonia lunar para un emperador romano.

Después de ejecutar estas celdas, observa cuidadosamente las respuestas. Fíjate en:

*   **Afirmaciones de hechos que son incorrectas.**
*   **Detalles inventados o ficcionalizados.**
*   **Respuestas que parecen plausibles pero no tienen fundamento en la realidad.**

Estos ejemplos nos darán una base para entender el problema y luego implementar las medidas anti-alucinación.

# Sin alucionaciones no podemos hacer anti alucinación

Como observamos los modelos nuevos tienen medidas anti-alucionacion incorporadas, sin embargo debemos recordar que estos modelos tienes un costo por lo que algunas veces optaremos por modelos locales o con menos potencia y debemos asegurarnos que sigan teniendo medias anti alucionación

## Configuración de Qwen3-4B vía Ollama

En vez del modelo original de este ejercicio (7B, con acceso restringido en Hugging Face, ~15GB, pensado para GPU de Colab), usamos **Qwen3-4B** servido por el contenedor `ollama` que ya está definido en `docker-compose.yml` (mismo patrón que `docs/Modulo 01/Proyecto_01/agent.py`). Es un modelo abierto y más chico, sin licencia gated, que corre razonablemente bien en CPU.

In [ ]:
# Ya no instalamos transformers/accelerate/bitsandbytes: Qwen3-4B corre en el
# contenedor `ollama` (servicio de docker-compose.yml), consumido por HTTP con
# la librería `requests`, ya declarada en el proyecto.

Nos conectamos al contenedor `ollama` (ya levantado por `docker-compose.yml`) y definimos dos shims con la misma interfaz que usa el resto del notebook (`tokenizer.apply_chat_template` y `mistral_pipeline(...)`), para no tener que tocar las celdas de abajo.

In [ ]:
import os

import requests

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL", "http://localhost:11434/api/chat")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:4b")


class _ChatTemplateTokenizer:
    """Shim minimo que reemplaza al tokenizer de Mistral/HF: solo formatea la
    lista de mensajes en un string, que Ollama recibe como prompt de chat."""

    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True):
        return "\n".join(f"[{m['role'].upper()}] {m['content']}" for m in messages)


def mistral_pipeline(formatted_prompt, max_new_tokens=200, do_sample=True,
                      temperature=0.7, top_k=50, top_p=0.95):
    """Shim con la misma firma que un pipeline de `transformers`, pero que llama
    a Qwen3-4B via la API local de Ollama en vez de cargar pesos en memoria.
    Devuelve el mismo formato ([{'generated_text': ...}]) para que el resto del
    notebook no necesite cambios."""
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": formatted_prompt}],
        "stream": False,
        "options": {
            "temperature": temperature if do_sample else 0.0,
            "top_k": top_k,
            "top_p": top_p,
            "num_predict": max_new_tokens,
        },
    }
    response = requests.post(OLLAMA_API_URL, json=payload, timeout=180)
    response.raise_for_status()
    content = response.json().get("message", {}).get("content", "")
    return [{"generated_text": formatted_prompt + content}]


tokenizer = _ChatTemplateTokenizer()
print(f"Cliente Ollama listo. Modelo: '{OLLAMA_MODEL}' en '{OLLAMA_API_URL}'.")

## Re-evaluando Alucinaciones con Qwen3-4B (vía Ollama)

Ahora que tenemos el modelo Qwen3-4B configurado, vamos a ejecutar los mismos `prompts` que usamos con Gemini para ver cómo responde este modelo y si es más propenso a las alucinaciones.

### Ejemplo 1: Pregunta sobre hechos oscuros o inexistentes (Qwen3-4B)

In [ ]:
prompt_1_mistral = load_prompt("nobel_literatura_1890")

# Formato de chat de Qwen3-4B
messages_1 = [
    {"role": "user", "content": prompt_1_mistral}
]

# Aplicar el formato de chat del tokenizador
formatted_prompt_1_mistral = tokenizer.apply_chat_template(messages_1, tokenize=False, add_generation_prompt=True)

# Generar respuesta usando el pipeline de Qwen3-4B
response_1_mistral = mistral_pipeline(
    formatted_prompt_1_mistral,
    max_new_tokens=200, # Limitar la longitud de la respuesta
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

# Extraer la respuesta del modelo, eliminando el prompt de la salida
model_output_1 = response_1_mistral[0]['generated_text']
# Buscar el final del prompt y tomar solo la parte generada
model_output_1 = model_output_1[len(formatted_prompt_1_mistral):].strip()

print(f"Pregunta: {prompt_1_mistral}\n")
print(f"Respuesta del modelo (Qwen3-4B):\n{model_output_1}")

### Ejemplo 2: Creación de detalles inexistentes para un objeto conocido (Qwen3-4B)

In [ ]:
prompt_2_mistral = load_prompt("uss_enterprise_d_propulsion")

# Formato de chat de Qwen3-4B
messages_2 = [
    {"role": "user", "content": prompt_2_mistral}
]

# Aplicar el formato de chat del tokenizador
formatted_prompt_2_mistral = tokenizer.apply_chat_template(messages_2, tokenize=False, add_generation_prompt=True)

# Generar respuesta usando el pipeline de Qwen3-4B
response_2_mistral = mistral_pipeline(
    formatted_prompt_2_mistral,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

# Extraer la respuesta del modelo, eliminando el prompt de la salida
model_output_2 = response_2_mistral[0]['generated_text']
# Buscar el final del prompt y tomar solo la parte generada
model_output_2 = model_output_2[len(formatted_prompt_2_mistral):].strip()

print(f"Pregunta: {prompt_2_mistral}\n")
print(f"Respuesta del modelo (Qwen3-4B):\n{model_output_2}")

### Ejemplo 3: Preguntas con premisas falsas (Qwen3-4B)

In [ ]:
prompt_3_mistral = load_prompt("julio_cesar_luna")

# Formato de chat de Qwen3-4B
messages_3 = [
    {"role": "user", "content": prompt_3_mistral}
]

# Aplicar el formato de chat del tokenizador
formatted_prompt_3_mistral = tokenizer.apply_chat_template(messages_3, tokenize=False, add_generation_prompt=True)

# Generar respuesta usando el pipeline de Qwen3-4B
response_3_mistral = mistral_pipeline(
    formatted_prompt_3_mistral,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

# Extraer la respuesta del modelo, eliminando el prompt de la salida
model_output_3 = response_3_mistral[0]['generated_text']
# Buscar el final del prompt y tomar solo la parte generada
model_output_3 = model_output_3[len(formatted_prompt_3_mistral):].strip()

print(f"Pregunta: {prompt_3_mistral}\n")
print(f"Respuesta del modelo (Qwen3-4B):\n{model_output_3}")

## Implementación de RAG (Retrieval-Augmented Generation)

Vamos a configurar un entorno básico para RAG. Para ello, necesitamos un "corpus" de información y una forma de buscar en él. Para simplificar, usaremos un diccionario de hechos y una búsqueda simple por palabra clave, o embedding si es posible con las librerías disponibles.

### Paso 1: Crear un Corpus de Conocimiento

Crearemos un pequeño "documento" o base de datos de hechos relevantes para las preguntas que hemos usado.

In [ ]:
knowledge_base = {
    "nobel_prize": "El Premio Nobel de Literatura fue establecido por Alfred Nobel y se otorgó por primera vez en 1901. No hubo Premio Nobel en el año 1890.",
    "uss_enterprise_d_propulsion": "Los sistemas de propulsión de la USS Enterprise D en Star Trek incluyen Warp Drive (motor de curvatura) e Impulse Engines (motores de impulso). No se menciona un 'Gravity Plating' ni 'quantum slipstream drive' como sistemas principales en las guías técnicas canónicas.",
    "julius_caesar": "Julio César fue un general y político romano que vivió entre el 100 a.C. y el 44 a.C. Nunca viajó a la Luna. La primera persona en pisar la Luna fue Neil Armstrong en 1969."
}

print("Base de conocimiento creada.")

### Paso 2: Función de Recuperación (Simple Retrieval)

Crearemos una función sencilla que, basándose en la pregunta, "recupere" el fragmento de información más relevante de nuestra `knowledge_base`.

In [ ]:
def retrieve_info(query, kb):
    # Esta es una función de recuperación muy básica.
    # En un sistema RAG real, se usarían embeddings y búsqueda vectorial.
    query_lower = query.lower()
    if "nobel" in query_lower and "1890" in query_lower:
        return kb["nobel_prize"]
    elif "enterprise d" in query_lower and "propulsion" in query_lower:
        return kb["uss_enterprise_d_propulsion"]
    elif "julio césar" in query_lower and "luna" in query_lower:
        return kb["julius_caesar"]
    return "No se encontró información relevante en la base de conocimientos."

print("Función de recuperación creada.")

### Paso 3: Aplicar RAG al Modelo Qwen3-4B

Ahora, combinaremos la recuperación con la generación del modelo Qwen3-4B. Usaremos el prompt de "USS Enterprise D" y "Julio César en la Luna", ya que fueron los que mostraron alucinaciones más claras.

#### Ejemplo RAG 1: USS Enterprise D con información recuperada

In [ ]:
prompt_2_rag = load_prompt("uss_enterprise_d_propulsion")

# Recuperar información
retrieved_context_2 = retrieve_info(prompt_2_rag, knowledge_base)

# Construir el prompt aumentado para Qwen3-4B
rag_prompt_2 = load_prompt("rag_context_wrapper").format(context=retrieved_context_2, prompt=prompt_2_rag)

messages_2_rag = [
    {"role": "user", "content": rag_prompt_2}
]

formatted_prompt_2_rag = tokenizer.apply_chat_template(messages_2_rag, tokenize=False, add_generation_prompt=True)

response_2_rag = mistral_pipeline(
    formatted_prompt_2_rag,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_2_rag = response_2_rag[0]['generated_text']
model_output_2_rag = model_output_2_rag[len(formatted_prompt_2_rag):].strip()

print(f"Pregunta con RAG: {prompt_2_rag}\n")
print(f"Contexto recuperado: {retrieved_context_2}\n")
print(f"Respuesta del modelo (Qwen3-4B con RAG):\n{model_output_2_rag}")

#### Ejemplo RAG 2: Julio César en la Luna con información recuperada

In [ ]:
prompt_3_rag = load_prompt("julio_cesar_luna")

# Recuperar información
retrieved_context_3 = retrieve_info(prompt_3_rag, knowledge_base)

# Construir el prompt aumentado para Qwen3-4B
rag_prompt_3 = load_prompt("rag_context_wrapper").format(context=retrieved_context_3, prompt=prompt_3_rag)

messages_3_rag = [
    {"role": "user", "content": rag_prompt_3}
]

formatted_prompt_3_rag = tokenizer.apply_chat_template(messages_3_rag, tokenize=False, add_generation_prompt=True)

response_3_rag = mistral_pipeline(
    formatted_prompt_3_rag,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_3_rag = response_3_rag[0]['generated_text']
model_output_3_rag = model_output_3_rag[len(formatted_prompt_3_rag):].strip()

print(f"Pregunta con RAG: {prompt_3_rag}\n")
print(f"Contexto recuperado: {retrieved_context_3}\n")
print(f"Respuesta del modelo (Qwen3-4B con RAG):\n{model_output_3_rag}")

Observa cómo las respuestas del modelo Qwen3-4B cambian ahora que tiene una "fuente de verdad" a la que referirse. Debería ser menos propenso a inventar detalles o a seguir premisas falsas.

## Implementación de Chain of Thought (CoT)

Vamos a aplicar la técnica CoT a los mismos prompts que usamos anteriormente con Qwen3-4B para ver si el modelo genera respuestas menos alucinadas al "pensar en voz alta".

### Ejemplo CoT 1: USS Enterprise D con Chain of Thought

In [ ]:
prompt_2_cot = load_prompt("uss_enterprise_d_propulsion") + " Piensa paso a paso y explica cómo llegas a tu respuesta antes de darla."

messages_2_cot = [
    {"role": "user", "content": prompt_2_cot}
]

formatted_prompt_2_cot = tokenizer.apply_chat_template(messages_2_cot, tokenize=False, add_generation_prompt=True)

response_2_cot = mistral_pipeline(
    formatted_prompt_2_cot,
    max_new_tokens=400, # Aumentamos tokens para el razonamiento
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_2_cot = response_2_cot[0]['generated_text']
model_output_2_cot = model_output_2_cot[len(formatted_prompt_2_cot):].strip()

print(f"Pregunta con CoT: {prompt_2_cot}\n")
print(f"Respuesta del modelo (Qwen3-4B con CoT):\n{model_output_2_cot}")

### Ejemplo CoT 2: Julio César en la Luna con Chain of Thought

In [ ]:
prompt_3_cot = load_prompt("julio_cesar_luna") + " Piensa paso a paso y explica tu razonamiento antes de dar la respuesta."

messages_3_cot = [
    {"role": "user", "content": prompt_3_cot}
]

formatted_prompt_3_cot = tokenizer.apply_chat_template(messages_3_cot, tokenize=False, add_generation_prompt=True)

response_3_cot = mistral_pipeline(
    formatted_prompt_3_cot,
    max_new_tokens=400, # Aumentamos tokens para el razonamiento
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_3_cot = response_3_cot[0]['generated_text']
model_output_3_cot = model_output_3_cot[len(formatted_prompt_3_cot):].strip()

print(f"Pregunta con CoT: {prompt_3_cot}\n")
print(f"Respuesta del modelo (Qwen3-4B con CoT):\n{model_output_3_cot}")

## Implementación de Self-consistency (Auto-consistencia)

Para Self-consistency, vamos a ejecutar el mismo prompt con CoT varias veces y luego compararemos las respuestas para ver su consistencia.

### Ejemplo Self-consistency 1: Julio César en la Luna (Ejecución 1)

In [ ]:
prompt_3_cot_sc = load_prompt("julio_cesar_luna") + " Piensa paso a paso y explica tu razonamiento antes de dar la respuesta."

messages_3_cot_sc_1 = [
    {"role": "user", "content": prompt_3_cot_sc}
]

formatted_prompt_3_cot_sc_1 = tokenizer.apply_chat_template(messages_3_cot_sc_1, tokenize=False, add_generation_prompt=True)

response_3_cot_sc_1 = mistral_pipeline(
    formatted_prompt_3_cot_sc_1,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_3_cot_sc_1 = response_3_cot_sc_1[0]['generated_text']
model_output_3_cot_sc_1 = model_output_3_cot_sc_1[len(formatted_prompt_3_cot_sc_1):].strip()

print(f"Pregunta (con CoT para SC): {prompt_3_cot_sc}\n")
print(f"--- Respuesta del modelo (Qwen3-4B con CoT y SC - Ejecución 1):---\n{model_output_3_cot_sc_1}\n")

### Ejemplo Self-consistency 1: Julio César en la Luna (Ejecución 2)

In [ ]:
prompt_3_cot_sc = load_prompt("julio_cesar_luna") + " Piensa paso a paso y explica tu razonamiento antes de dar la respuesta."

messages_3_cot_sc_2 = [
    {"role": "user", "content": prompt_3_cot_sc}
]

formatted_prompt_3_cot_sc_2 = tokenizer.apply_chat_template(messages_3_cot_sc_2, tokenize=False, add_generation_prompt=True)

response_3_cot_sc_2 = mistral_pipeline(
    formatted_prompt_3_cot_sc_2,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_3_cot_sc_2 = response_3_cot_sc_2[0]['generated_text']
model_output_3_cot_sc_2 = model_output_3_cot_sc_2[len(formatted_prompt_3_cot_sc_2):].strip()

print(f"Pregunta (con CoT para SC): {prompt_3_cot_sc}\n")
print(f"--- Respuesta del modelo (Qwen3-4B con CoT y SC - Ejecución 2):---\n{model_output_3_cot_sc_2}\n")

### Ejemplo Self-consistency 1: Julio César en la Luna (Ejecución 3)

In [ ]:
prompt_3_cot_sc = load_prompt("julio_cesar_luna") + " Piensa paso a paso y explica tu razonamiento antes de dar la respuesta."

messages_3_cot_sc_3 = [
    {"role": "user", "content": prompt_3_cot_sc}
]

formatted_prompt_3_cot_sc_3 = tokenizer.apply_chat_template(messages_3_cot_sc_3, tokenize=False, add_generation_prompt=True)

response_3_cot_sc_3 = mistral_pipeline(
    formatted_prompt_3_cot_sc_3,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

model_output_3_cot_sc_3 = response_3_cot_sc_3[0]['generated_text']
model_output_3_cot_sc_3 = model_output_3_cot_sc_3[len(formatted_prompt_3_cot_sc_3):].strip()

print(f"Pregunta (con CoT para SC): {prompt_3_cot_sc}\n")
print(f"--- Respuesta del modelo (Qwen3-4B con CoT y SC - Ejecución 3):---\n{model_output_3_cot_sc_3}\n")

## Implementación del Ajuste de Temperatura

Vamos a probar cómo la `temperature` afecta las respuestas del modelo. Usaremos el prompt del USS Enterprise D, que fue propenso a alucinaciones sin RAG ni un CoT muy específico para refutar la premisa.

### Ejemplo Temperatura 1: USS Enterprise D (Temperatura Alta)

Aquí configuraremos una temperatura más alta (por ejemplo, 0.9) para ver si el modelo es más propenso a alucinar o a ser más creativo con sus respuestas.

In [ ]:
prompt_2_temp_high = load_prompt("uss_enterprise_d_propulsion")

messages_2_temp_high = [
    {"role": "user", "content": prompt_2_temp_high}
]

formatted_prompt_2_temp_high = tokenizer.apply_chat_template(messages_2_temp_high, tokenize=False, add_generation_prompt=True)

response_2_temp_high = mistral_pipeline(
    formatted_prompt_2_temp_high,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.9, # Temperatura alta
    top_k=50,
    top_p=0.95
)

model_output_2_temp_high = response_2_temp_high[0]['generated_text']
model_output_2_temp_high = model_output_2_temp_high[len(formatted_prompt_2_temp_high):].strip()

print(f"Pregunta (Temp Alta): {prompt_2_temp_high}\n")
print(f"--- Respuesta del modelo (Qwen3-4B - Temp Alta):---\n{model_output_2_temp_high}\n")

### Ejemplo Temperatura 2: USS Enterprise D (Temperatura Baja)

Ahora, configuraremos una temperatura más baja (por ejemplo, 0.1) para ver si el modelo se vuelve más conservador y menos propenso a inventar detalles sobre la 'Guía Técnica del Universo Star Trek Volumen 5'.

In [ ]:
prompt_2_temp_low = load_prompt("uss_enterprise_d_propulsion")

messages_2_temp_low = [
    {"role": "user", "content": prompt_2_temp_low}
]

formatted_prompt_2_temp_low = tokenizer.apply_chat_template(messages_2_temp_low, tokenize=False, add_generation_prompt=True)

response_2_temp_low = mistral_pipeline(
    formatted_prompt_2_temp_low,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.1, # Temperatura baja
    top_k=50,
    top_p=0.95
)

model_output_2_temp_low = response_2_temp_low[0]['generated_text']
model_output_2_temp_low = model_output_2_temp_low[len(formatted_prompt_2_temp_low):].strip()

print(f"Pregunta (Temp Baja): {prompt_2_temp_low}\n")
print(f"--- Respuesta del modelo (Qwen3-4B - Temp Baja):---\n{model_output_2_temp_low}\n")

Observa las diferencias en las respuestas. Con una temperatura más baja, el modelo debería ser menos propenso a inventar información, y quizás incluso admita que la fuente no existe o que no puede proporcionar detalles tan específicos.

Después de ejecutar las celdas, observa si las tres respuestas son consistentes en su corrección de la premisa falsa o si alguna de ellas alucina de manera diferente. La consistencia en la negación de la premisa falsa es un signo de mayor fiabilidad.

Observa cómo el modelo Qwen3-4B genera un proceso de pensamiento antes de intentar responder. Esto debería ayudarle a identificar inconsistencias o la falsedad de la premisa, reduciendo las alucinaciones.